# Задача 2, этап 2: обучение до схождения

В первой версии (`task2.ipynb`) PPO и DSAC получили разное число эпизодов, и DSAC на картах 10x10 не доучился.
Здесь каждый прогон учится, пока не выйдет на плато (или не упрётся в потолок эпизодов), — так мы узнаём,
сколько шагов среды нужно каждому алгоритму. Гиперпараметры те же, seed 0. Результаты — `outputs/task2_v2/`.

По этим точкам схождения выбирается одинаковый бюджет шагов для честного сравнения в `task2_v3.ipynb`.

In [1]:
import csv
import json
import os
import sys
from pathlib import Path

# работаем из корня репозитория: там лежат src/ и outputs/
if Path.cwd().name == "tasks":
    os.chdir(Path.cwd().parent)
sys.path.insert(0, str(Path.cwd()))

import matplotlib.pyplot as plt
import numpy as np
import torch

from src import utils
from src.envs import load_mnist, make_env
from src.models import DSAC, PPO

In [2]:
# режим запуска: при обычном открытии ноутбука — полное обучение всех экспериментов и все графики
# на сервере скрипт задаёт переменные, чтобы запускать эксперименты параллельно
RUN_MODE = os.environ.get("RUN_MODE", "full")  # full или smoke (короткая проверка)
RUN_ONLY = os.environ.get("RUN_ONLY", "")  # имя одного эксперимента, пусто — все
RUN_STAGES = os.environ.get("RUN_STAGES", "train,plots").split(",")

torch.set_num_threads(1)  # прогоны идут параллельно, каждому хватит одного потока
CUDA = torch.cuda.is_available()
print("режим:", RUN_MODE, "| эксперимент:", RUN_ONLY or "все", "| этапы:", RUN_STAGES, "| CUDA:", CUDA)

режим: full | эксперимент: dsac_env4_mlp_seed0 | этапы: ['train'] | CUDA: True


In [3]:
SETUPS = {
    1: dict(h=5, w=5, n_colors=None, obstacle_fraction=0.0),  # 25 уникальных цветов — полностью наблюдаемая
    2: dict(h=5, w=5, n_colors=5, obstacle_fraction=0.0),
    3: dict(h=10, w=10, n_colors=7, obstacle_fraction=0.1),
    4: dict(h=10, w=10, n_colors=4, obstacle_fraction=0.1),
}

## Гиперпараметры

In [4]:
ppo_cfg = {
    "device": "cpu",  # PPO на видеокарте не быстрее (замер: 479 против 429 шагов в секунду)
    "env_seed": 0,  # от него зависит карта
    "seed": 0,  # от него зависят сеть и старты эпизодов
    "memory": True,  # True — GRU, False — MLP без памяти
    "episodes_per_update": 10,  # сколько эпизодов собрать перед обновлением
    "epochs": 4,
    "episodes_per_batch": 5,
    "gamma": 0.99,
    "gae_lambda": 0.95,
    "lr": 3e-4,
    "clip": 0.2,
    "ent_coef": 0.01,
    "vf_coef": 0.5,
    "hidden": 128,
    "save_every": 50,  # как часто сохранять метрики и веса на диск
}

In [5]:
dsac_cfg = {
    "device": "cuda" if CUDA else "cpu",  # DSAC на видеокарте в 3 раза быстрее (167 против 52 шагов в секунду)
    "env_seed": 0,
    "seed": 0,
    "memory": True,
    "soft_q": True,  # мягкая Q (энтропия в цели критика) или обычная
    "buffer_episodes": 5000,  # сколько последних эпизодов помнить
    "learning_starts": 20,  # сколько эпизодов только копим опыт, не учась
    "updates_per_episode": 4,  # сколько обновлений после каждого эпизода
    "batch_size": 8,  # сколько случайных эпизодов из буфера в одном обновлении
    "gamma": 0.99,
    "tau": 0.005,  # скорость, с которой target-критики догоняют обычных
    "lr": 3e-4,
    "alpha": 0.005,  # температура: вес энтропии
    "hidden": 128,
    "save_every": 25,
}

In [6]:
# учимся до выхода на плато: каждые 100 эпизодов сравниваем среднюю отдачу за последние conv_window эпизодов
# с предыдущими; если прирост меньше conv_tol два раза подряд — остановка.
# Потолок эпизодов — на случай, если алгоритм так и не сойдётся (тогда в status.json будет converged = false)
MAX_EPISODES = {"full": {1: 4000, 2: 4000, 3: 8000, 4: 8000}, "smoke": {1: 30, 2: 30, 3: 20, 4: 20}}
# на картах 10x10 плато ищем только после 2000 эпизодов: DSAC сначала долго «стоит» около нуля,
# и без этой паузы критерий принимал застой за схождение
MIN_EPISODES = {"full": {1: 0, 2: 0, 3: 2000, 4: 2000}, "smoke": {1: 0, 2: 0, 3: 0, 4: 0}}
CONVERGENCE = {
    "full": {"stop_when_converged": True, "conv_window": 200, "conv_tol": 0.01},
    "smoke": {"stop_when_converged": True, "conv_window": 5, "conv_tol": 0.01},
}

EVAL_EPISODES = {"full": 100, "smoke": 5}

OUT = "outputs/task2_v2" if RUN_MODE == "full" else "runs/smoke/task2_v2"

## Список экспериментов

In [7]:
EXPERIMENTS = []
for seed in [0]:
    for i in SETUPS:
        for memory in [True, False]:
            tag = "gru" if memory else "mlp"
            for algo, base in [("ppo", ppo_cfg), ("dsac", dsac_cfg)]:
                name = f"{algo}_env{i}_{tag}_seed{seed}"
                cfg = {**base, **SETUPS[i], "memory": memory, "seed": seed, "episodes": MAX_EPISODES[RUN_MODE][i], **CONVERGENCE[RUN_MODE], "min_episodes": MIN_EPISODES[RUN_MODE][i],
                       "budget": RUN_MODE, "out_dir": f"{OUT}/runs", "run_name": name}
                EXPERIMENTS.append({"name": name, "algo": algo, "cfg": cfg})

    # требование задания: DSAC с обычной Q вместо мягкой
    for i in [2, 4]:
        name = f"dsac_env{i}_gru_hardq_seed{seed}"
        cfg = {**dsac_cfg, **SETUPS[i], "soft_q": False, "seed": seed, "episodes": MAX_EPISODES[RUN_MODE][i], **CONVERGENCE[RUN_MODE], "min_episodes": MIN_EPISODES[RUN_MODE][i],
               "budget": RUN_MODE, "out_dir": f"{OUT}/runs", "run_name": name}
        EXPERIMENTS.append({"name": name, "algo": "dsac", "cfg": cfg})

for exp in EXPERIMENTS:
    print(exp["name"], exp["cfg"]["device"])

ppo_env1_gru_seed0 cpu
dsac_env1_gru_seed0 cuda
ppo_env1_mlp_seed0 cpu
dsac_env1_mlp_seed0 cuda
ppo_env2_gru_seed0 cpu
dsac_env2_gru_seed0 cuda
ppo_env2_mlp_seed0 cpu
dsac_env2_mlp_seed0 cuda
ppo_env3_gru_seed0 cpu
dsac_env3_gru_seed0 cuda
ppo_env3_mlp_seed0 cpu
dsac_env3_mlp_seed0 cuda
ppo_env4_gru_seed0 cpu
dsac_env4_gru_seed0 cuda
ppo_env4_mlp_seed0 cpu
dsac_env4_mlp_seed0 cuda
dsac_env2_gru_hardq_seed0 cuda
dsac_env4_gru_hardq_seed0 cuda


## Обучение и оценка

In [8]:
def run_experiment(exp):
    # прогон, который уже закончился, второй раз не учим
    status_file = Path(OUT) / "runs" / exp["name"] / "status.json"
    if status_file.exists() and json.loads(status_file.read_text())["status"] == "completed":
        print("уже готово:", exp["name"])
        return

    if exp["algo"] == "ppo":
        alg = PPO(exp["cfg"])
    else:
        alg = DSAC(exp["cfg"])
    alg.train()

    # оценка на одних и тех же стартах (сиды 10000, 10001, ...): жадно и с сэмплированием действий
    n = EVAL_EPISODES[RUN_MODE]
    greedy = alg.evaluate(n, greedy=True)
    sampled = alg.evaluate(n, greedy=False)
    result = {"budget": RUN_MODE,
              "greedy": utils.eval_summary(greedy, alg.cfg["gamma"], alg.optimal),
              "stochastic": utils.eval_summary(sampled, alg.cfg["gamma"], alg.optimal)}
    utils.save_eval(alg.run_dir, result)

    s = result["stochastic"]
    title = f"{exp['name']}: успех {s['success_rate']:.2f}, длина {s['mean_length']:.1f} (оптимум {s['optimal_length']:.1f})"
    utils.save_eval_pictures(alg.env, sampled, Path(OUT) / "eval", exp["name"], title)

In [9]:
if "train" in RUN_STAGES:
    for exp in EXPERIMENTS:
        if RUN_ONLY == "" or RUN_ONLY == exp["name"]:
            run_experiment(exp)

episode 50 | steps 16007 | return 0.118 (opt 0.938) | length 320.1 | entropy 1.08


episode 100 | steps 31721 | return 0.153 (opt 0.938) | length 314.3 | entropy 0.79


episode 150 | steps 44413 | return 0.332 (opt 0.938) | length 253.8 | entropy 0.38


episode 200 | steps 60207 | return 0.170 (opt 0.938) | length 315.9 | entropy 0.29


episode 250 | steps 78295 | return 0.084 (opt 0.938) | length 361.8 | entropy 0.24


...

episode 2250 | steps 692940 | return 0.126 (opt 0.938) | length 305.2 | entropy 0.89


episode 2300 | steps 706605 | return 0.194 (opt 0.938) | length 273.3 | entropy 0.95


episode 2350 | steps 718809 | return 0.275 (opt 0.938) | length 244.1 | entropy 0.91


вышли на плато: эпизод 2400, шагов 733097
episode 2400 | steps 733097 | return 0.175 (opt 0.938) | length 285.8 | entropy 0.90


saved outputs/task2_v2/eval/dsac_env4_mlp_seed0.gif


## Когда алгоритмы сошлись

In [10]:
plt.style.use("seaborn-v0_8-darkgrid")


def load_metrics(run_path):
    # metrics.csv прогона: словарь {колонка: массив чисел}; None, если прогона нет
    path = Path(run_path) / "metrics.csv"
    if not path.exists():
        print("нет данных:", run_path)
        return None
    with open(path, encoding="utf-8") as f:
        rows = list(csv.DictReader(f))
    data = {}
    for key in rows[0]:
        values = [{"True": 1.0, "False": 0.0, "": np.nan}.get(row[key], row[key]) for row in rows]
        data[key] = np.array(values, dtype=float)
    return data

In [11]:
def convergence_rows(run_paths):
    # когда каждый прогон вышел на плато (или упёрся в потолок эпизодов) и как он потом сыграл в оценке
    rows = []
    for path in run_paths:
        status_file = Path(path) / "status.json"
        if not status_file.exists():
            rows.append({"run": Path(path).name, "converged": "pending"})
            continue
        status = json.loads(status_file.read_text(encoding="utf-8"))
        row = {"run": Path(path).name, "converged": status.get("converged"),
               "episodes": status["episodes_done"], "env_steps": status.get("env_steps"),
               "time_min": round(status.get("time_sec", 0) / 60, 1)}
        eval_file = Path(path) / "eval.json"
        if eval_file.exists():
            result = json.loads(eval_file.read_text(encoding="utf-8"))
            row["greedy_success"] = round(result["greedy"]["success_rate"], 3)
            row["greedy_length"] = round(result["greedy"]["mean_length"], 1)
            row["optimal_length"] = round(result["greedy"]["optimal_length"], 2)
        rows.append(row)
    return rows


def save_rows(rows, filename):
    Path(OUT, "tables").mkdir(parents=True, exist_ok=True)
    fields = []
    for row in rows:
        for key in row:
            if key not in fields:
                fields.append(key)
    with open(Path(OUT) / "tables" / filename, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fields)
        writer.writeheader()
        writer.writerows(rows)
    for row in rows:
        print(row)

In [12]:
if "plots" in RUN_STAGES:
    save_rows(convergence_rows([f"{OUT}/runs/{exp['name']}" for exp in EXPERIMENTS]), "convergence.csv")

## Графики обучения по шагам среды

In [13]:
def rolling_mean(values, window=50):
    # среднее за последние window эпизодов (пропуски nan не учитываем)
    result = []
    for i in range(len(values)):
        last = values[max(0, i - window + 1):i + 1]
        last = last[~np.isnan(last)]
        result.append(np.mean(last) if len(last) > 0 else np.nan)
    return np.array(result)

In [14]:
def plot_compare_steps(runs, labels, key, ylabel, filename):
    # как в задаче 2, но по оси X — шаги среды; линия — скользящее среднее за 50 эпизодов (1 сид)
    colors = ["tab:red", "tab:blue", "tab:green"]
    fig, axes = plt.subplots(1, len(runs), figsize=(5.5 * len(runs), 4.3), squeeze=False)
    for ax, env_name in zip(axes[0], runs):
        for path, label, color in zip(runs[env_name], labels, colors):
            data = load_metrics(path)
            if data is not None:
                ax.plot(data["env_steps"], rolling_mean(data[key]), color=color, linewidth=2, label=label)
        ax.set_title(env_name)
        ax.set_xlabel("шаги среды")
        ax.set_ylabel(ylabel)
        ax.legend(fontsize=8)
    Path(OUT, "figures").mkdir(parents=True, exist_ok=True)
    fig.savefig(Path(OUT) / "figures" / filename, dpi=150, bbox_inches="tight")
    plt.show()

In [15]:
if "plots" in RUN_STAGES:
    runs = {f"Env{i}": [f"{OUT}/runs/ppo_env{i}_gru_seed0", f"{OUT}/runs/dsac_env{i}_gru_seed0"] for i in SETUPS}
    for key, ylabel in [("return", "средняя дисконтированная отдача"), ("success", "доля успешных эпизодов"),
                        ("length", "длина эпизода"), ("entropy", "энтропия стратегии"),
                        ("explore_prob", "вероятность нежадного действия"), ("loss", "полный лосс")]:
        plot_compare_steps(runs, ["PPO + GRU", "DSAC + GRU"], key, ylabel, f"ppo_vs_dsac_{key}.png")

In [16]:
if "plots" in RUN_STAGES:
    for algo in ["ppo", "dsac"]:
        runs = {f"Env{i}": [f"{OUT}/runs/{algo}_env{i}_gru_seed0", f"{OUT}/runs/{algo}_env{i}_mlp_seed0"] for i in SETUPS}
        plot_compare_steps(runs, ["GRU", "MLP"], "return", "средняя дисконтированная отдача", f"{algo}_memory_return.png")
    runs = {f"Env{i}": [f"{OUT}/runs/dsac_env{i}_gru_seed0", f"{OUT}/runs/dsac_env{i}_gru_hardq_seed0"] for i in [2, 4]}
    plot_compare_steps(runs, ["soft_q=True", "soft_q=False"], "return", "средняя дисконтированная отдача", "dsac_soft_q_return.png")

## Итоговая оценка

In [17]:
def eval_rows(run_paths):
    # строки итоговой таблицы из eval.json прогонов
    rows = []
    for path in run_paths:
        eval_file = Path(path) / "eval.json"
        if not eval_file.exists():
            rows.append({"run": Path(path).name, "eval": "pending"})
            continue
        result = json.loads(eval_file.read_text(encoding="utf-8"))
        for mode in ["greedy", "stochastic"]:
            r = result[mode]
            rows.append({"run": Path(path).name, "eval": mode, "success_rate": round(r["success_rate"], 3),
                         "mean_return": round(r["mean_return"], 3), "mean_length": round(r["mean_length"], 1),
                         "failures": r["failures"], "optimal_length": round(r["optimal_length"], 2)})
    return rows


def save_table(rows, filename):
    Path(OUT, "tables").mkdir(parents=True, exist_ok=True)
    fields = ["run", "eval", "success_rate", "mean_return", "mean_length", "failures", "optimal_length"]
    with open(Path(OUT) / "tables" / filename, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fields)
        writer.writeheader()
        writer.writerows(rows)
    for row in rows:
        print(row)

In [18]:
if "plots" in RUN_STAGES:
    save_table(eval_rows([f"{OUT}/runs/{exp['name']}" for exp in EXPERIMENTS]), "eval_summary.csv")